# 02 — Tournament analytics visualizations

Reads outputs from `data/processed/tournament_analytics/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
TA = ROOT / "data" / "processed" / "tournament_analytics"
hr = pd.read_parquet(TA / "historical_reference.parquet")
sp = pd.read_parquet(TA / "seed_pair_win_rates.parquet")
tt = pd.read_parquet(ROOT / "data" / "raw" / "torvik" / "tournament_training_set.parquet")

## Heatmap: better-seed win rate by round (parseable seed games only)

In [ ]:
piv = sp.pivot_table(
    index="seed_low", columns="round", values="historical_win_rate", aggfunc="mean"
)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(piv.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(piv.columns)))
ax.set_xticklabels(piv.columns, rotation=45, ha="right")
ax.set_yticks(range(len(piv.index)))
ax.set_yticklabels(piv.index)
ax.set_title("Historical win rate of better seed (subset with parseable 1–16 seeds)")
plt.colorbar(im, ax=ax, label="win rate")
plt.tight_layout()
plt.show()

## Bar: upset count by year (efficiency-based favorite)

In [ ]:
sub = hr.sort_values("season")
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(sub["season"].astype(str), sub["total_upsets"])
ax.set_xlabel("season")
ax.set_ylabel("total upsets")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Line: average margin by season (chaos proxy)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hr["season"], hr["avg_margin"], marker="o")
ax.set_xlabel("season")
ax.set_ylabel("avg margin")
plt.tight_layout()
plt.show()

## Stacked bar: conference wins (from JSON column)

In [ ]:
rows = []
for _, r in hr.iterrows():
    d = json.loads(r["conf_wins_json"]) if pd.notna(r["conf_wins_json"]) else {}
    for k, v in d.items():
        rows.append({"season": r["season"], "conf": k, "wins": v})
cw = pd.DataFrame(rows)
if len(cw):
    top = cw.groupby("conf")["wins"].sum().nlargest(8).index
    cw = cw[cw["conf"].isin(top)]
    piv2 = cw.pivot_table(index="season", columns="conf", values="wins", fill_value=0)
    piv2.plot(kind="bar", stacked=True, figsize=(12, 5), legend="reverse")
    plt.title("Tournament wins by conference (top 8 overall)")
    plt.tight_layout()
    plt.show()

## Table: summary stats (from summary parquet)

In [ ]:
sm = pd.read_parquet(TA / "historical_reference_summary.parquet")
print(sm)